# cAPTure: versioned graph OOF logit rescoring

This notebook repeats outer-fold inference from the eight immutable final checkpoints for Edge MLP, EdgeGRU, StaticGNN, and ST-GNN. It preserves the original float32 probability, the recomputed float32 probability, the raw model logit, and a sigmoid computed in float64.

No model is trained or refitted, no optimizer is created, and no threshold is selected. The original training OOF files, comparison, and saturation audit remain unchanged. Test1, Test2, `train_pub_exf`, and `train_user_prop` remain prohibited.

## 1. Mount Drive and prepare a GPU runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_rescoring_inputs")
LOCAL_WORK_ROOT = Path("/content/capture_graph_rescoring_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
import torch
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_oof_rescoring import (
    expected_rescoring_job_ids,
    finalize_capture_graph_oof_rescoring,
    load_graph_oof_rescoring_config,
    run_capture_graph_oof_rescoring_job,
    validate_capture_graph_oof_rescoring,
)
from utils.capture_graph_score_saturation import (
    validate_graph_score_saturation_audit,
)

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before rescoring.")
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("CUDA device:", torch.cuda.get_device_name(0))

## 2. Bind the immutable source artifacts

In [ ]:
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
TRAINING_BINDING_RUN_ID = (
    "20260929T144234_934044Z_graph_training_binding"
)
COMPARISON_RUN_ID = (
    "20261001T153351_608797Z_graph_four_model_comparison"
)
SATURATION_AUDIT_RUN_ID = (
    "20261001T175306_830261Z_graph_score_saturation_audit"
)
RESCORING_RUN_ID = "20261001_graph_oof_logit_rescoring_v1"

GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
TRAINING_ROOT = (
    DRIVE_ROOT / "graph_training_runs" / TRAINING_BINDING_RUN_ID
)
COMPARISON_DIR = (
    DRIVE_ROOT / "graph_comparison_runs" / COMPARISON_RUN_ID
)
SATURATION_AUDIT_DIR = (
    DRIVE_ROOT / "graph_score_saturation_audit_runs"
    / SATURATION_AUDIT_RUN_ID
)
RESCORING_OUTPUT_ROOT = (
    DRIVE_ROOT / "graph_oof_rescoring_runs" / RESCORING_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
RESCORING_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_oof_rescoring_v1.yaml"
)

for path in (
    GRAPH_MATERIALIZATION_DIR, PREPARED_RUN_DIR, TRAINING_ROOT,
    COMPARISON_DIR, SATURATION_AUDIT_DIR, GRAPH_INPUT_CONTRACT_PATH,
    RESCORING_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required input is missing: {path}")

rescoring_config = load_graph_oof_rescoring_config(RESCORING_CONFIG_PATH)
saturation_audit = validate_graph_score_saturation_audit(
    comparison_dir=COMPARISON_DIR,
    output_dir=SATURATION_AUDIT_DIR,
)
print("Saturation-audit status:", saturation_audit["status"])
print("Models requiring score-resolution correction:",
      saturation_audit["saturated_models"])
print("Rescoring output:", RESCORING_OUTPUT_ROOT)

## 3. Stage and checksum-verify the complete graph collection

Only the complete-fold graph materialization is needed. Selection features are not loaded because no epoch selection or training occurs.

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified graph collection:", collection.root)
print("All graph artifact checksums verified:",
      collection.artifact_checksums_verified)

## 4. Review and authorize the eight inference-only jobs

The default list contains all four models and both folds. Completed jobs are immutable and are reused on a rerun, so the same cell safely resumes after a runtime interruption. Set `RUN_RESCORING=True` only after checking the paths and job list.

In [ ]:
ALL_JOB_IDS = expected_rescoring_job_ids(rescoring_config)
JOB_IDS_TO_RUN = list(ALL_JOB_IDS)
RUN_RESCORING = False

job_table = pd.DataFrame([
    {
        "job_id": job_id,
        "model": job_id.rsplit("__fold_", 1)[0],
        "fold": job_id.rsplit("__fold_", 1)[1],
        "outer_validation_scenarios": rescoring_config["folds"][
            job_id.rsplit("__fold_", 1)[1]
        ]["outer_validation"],
    }
    for job_id in ALL_JOB_IDS
])
display(job_table)
if not JOB_IDS_TO_RUN or not set(JOB_IDS_TO_RUN).issubset(ALL_JOB_IDS):
    raise ValueError("JOB_IDS_TO_RUN contains an unknown or empty selection.")
print("Jobs selected:", JOB_IDS_TO_RUN)
print("Set RUN_RESCORING=True to begin or resume inference.")

## 5. Run or resume checkpoint rescoring

Each scenario is committed independently to Drive. If the runtime disconnects, rerun sections 1–4 with the same `RESCORING_RUN_ID`, leave the same jobs selected, and execute this cell again. Completed scenarios and jobs will be checksum-verified and reused. The run is sealed automatically only when all eight jobs are complete.

In [ ]:
if not RUN_RESCORING:
    print("Rescoring is disabled. No checkpoint was loaded.")
else:
    job_completions = []
    for job_id in JOB_IDS_TO_RUN:
        print(f"\nStarting or resuming {job_id}...")
        completion = run_capture_graph_oof_rescoring_job(
            job_id=job_id,
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            training_root=TRAINING_ROOT,
            prepared_run_dir=PREPARED_RUN_DIR,
            collection=collection,
            output_root=RESCORING_OUTPUT_ROOT,
            local_work_root=LOCAL_WORK_ROOT,
        )
        job_completions.append(completion)
    display(pd.DataFrame([{
        "job_id": item["job_id"],
        "status": item["status"],
        "model_training_performed": item["model_training_performed"],
        "held_out_scenarios_accessed": item["held_out_scenarios_accessed"],
    } for item in job_completions]))

## 6. Seal and review the completed run

This section seals the run only after every expected job exists. If only a subset was requested, it reports the remaining jobs without changing the completed artifacts.

In [ ]:
completed_job_ids = [
    job_id for job_id in ALL_JOB_IDS
    if (RESCORING_OUTPUT_ROOT / job_id / "completion.json").is_file()
]
remaining_job_ids = [
    job_id for job_id in ALL_JOB_IDS if job_id not in completed_job_ids
]
print("Completed jobs:", completed_job_ids)
print("Remaining jobs:", remaining_job_ids)

if remaining_job_ids:
    print("The rescoring run remains open until all eight jobs are complete.")
else:
    if (RESCORING_OUTPUT_ROOT / "run_status.json").is_file():
        rescoring_manifest = validate_capture_graph_oof_rescoring(
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            output_root=RESCORING_OUTPUT_ROOT,
        )
    else:
        rescoring_manifest = finalize_capture_graph_oof_rescoring(
            config_path=RESCORING_CONFIG_PATH,
            comparison_dir=COMPARISON_DIR,
            saturation_audit_dir=SATURATION_AUDIT_DIR,
            training_root=TRAINING_ROOT,
            prepared_run_dir=PREPARED_RUN_DIR,
            collection=collection,
            output_root=RESCORING_OUTPUT_ROOT,
        )
    display(pd.DataFrame(rescoring_manifest["jobs"]).T)
    scenario_rows = []
    for job_id in ALL_JOB_IDS:
        job_report = json.loads(
            (RESCORING_OUTPUT_ROOT / job_id / "job_report.json").read_text(
                encoding="utf-8"
            )
        )
        for scenario, metrics in job_report["scenarios"].items():
            scenario_rows.append({
                "model": job_report["model"],
                "fold": job_report["fold"],
                "scenario": scenario,
                "edges": metrics["edges"],
                "raw_logit_ap": metrics["average_precision_raw_logit"],
                "float64_score_ap": metrics["average_precision_score_float64"],
                "source_exact_match_fraction": metrics["source_probability_exact_match_fraction"],
                "source_maximum_absolute_difference": metrics["source_probability_maximum_absolute_difference"],
                "source_exact_one_count": metrics["source_probability_exact_one_count"],
                "float64_exact_one_count": metrics["score_float64_exact_one_count"],
                "raw_logit_minimum": metrics["raw_logit_minimum"],
                "raw_logit_maximum": metrics["raw_logit_maximum"],
            })
    display(pd.DataFrame(scenario_rows))
    print("Rescoring status:", rescoring_manifest["status"])
    print("Operational ranking field:",
          rescoring_manifest["operational_ranking_field"])
    print("Model training performed:",
          rescoring_manifest["model_training_performed"])
    print("Threshold selected:", rescoring_manifest["threshold_selected"])
    print("Held-out scenarios accessed:",
          rescoring_manifest["held_out_scenarios_accessed"])
    print("Next action:", rescoring_manifest["next_action"])